docker run -d -p 6333:6333 qdrant/qdrant
uv add qdrant-client      /    pip install qdrant-client

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct,Filter,FieldCondition,MatchValue

In [ ]:
# 准备数据
items = [
    {
        "type": "phone",
        "id": "用户A",
        "content": "用户A的手机号是13800001234"
    },
    {
        "type": "phone",
        "id": "用户B",
        "content": "用户B的手机号是13800005678"
    },
    {
        "type": "order",
        "id": "订单1001",
        "content": "订单1001编号203011010001"
    },
    {
        "type": "order",
        "id": "订单1002",
        "content": "订单1002编号203011010123"
    }
]

In [ ]:
 # 1. 连接 Qdrant
client = QdrantClient(
    url="http://localhost:6333"
)

In [ ]:
# 2. 创建 Collection
collection_name = "numbers_collection"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(            # 向量配置的容器：具体的向量参数设定类
                size=1,                         # 向量维度（必须与 Embedding 模型输出一致）
                distance=Distance.EUCLID        # 向量间距离的计算方法（决定了 Qdrant 怎么判断两个向量“有多近”）
            )
    )

In [ ]:
 # 3. 准备向量数据
points = []

for idx, item in enumerate(items):
    # 教学方便：直接把数字当作 1 维向量，实际应用和之前写的一样，用embedding向量
    vector = [float(item["number"])]

    points.append(
        PointStruct(                           # PointStruct：Qdrant 中写入数据的最小单元，把一条数据的所有相关信息打包在一起，插入到集合中
            id=idx,                            # 唯一标识符
            vector=vector,                     # 向量数据
            payload={                          # 存放向量的业务元数据/附加信息（类似Chroma的metadata）
                "type": item["type"],
                "item_id": item["id"],
                "number": item["number"]
            }
        )
    )

In [ ]:
# 4. upsert：不存在就插入，存在就更新
client.upsert(
        collection_name=collection_name,
        points=points
    )
print(f"已写入 {len(points)} 条数据")

In [ ]:
# 5. 查询
query_number = 205001020500
query_vector = [float(query_number)]

k = 5

results = client.query_points(            # query_points：搜索方法。它不仅能做向量搜索、推荐，还能做纯过滤查询
    collection_name=collection_name,
    query=query_vector,
    limit=k,
    with_payload=True,
)

print(f"\n查询数字:{query_number}")
print(f"Top-{k} 最相近的数据：")

for rank, hit in enumerate(results.points):
    print(
        f"第{rank}个："
        f"score={hit.score:.4f},"
        f"类型={hit.payload['type']},"
        f"编号={hit.payload['item_id']},"
        f"数字={hit.payload['number']}"
    )

In [ ]:
# 6. 带条件过滤的向量搜索
filtered_results = client.query_points(             # 相似度搜索
    collection_name=collection_name,
    query=query_vector,
    limit=k,
    with_payload=True,                               # 有了 payload，就可以做带条件的相似搜索
    query_filter=Filter(                             # 过滤条件
        must=[                                       # must：必须满足，相当于 SQL 中的 AND
            FieldCondition(                          # 匹配哪个字段，相当于 SQL 里的 字段名 = 值
                key="type",
                match=MatchValue(value="order")      # MatchValue：匹配规则
            )
        ]
    )
)
print("\n只搜索 type=order 的数据：")

for rank, hit in enumerate(filtered_results.points, start=1):
    print(
        f"第{rank}个："
        f"score={hit.score:.4f},"
        f"编号={hit.payload['item_id']},"
        f"数字={hit.payload['number']}"
    )